# Clasificador de texto generado por IA (español, tesis de educación)

Fine-tuning de un RoBERTa en español sobre segmentos de prosa de 150-250 palabras
(`dataset_clasificador_ia.zip`, exportado con `python manage.py exportar_dataset_ia`).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

Train de IA: textos generados por API (Gemini flash/flash-lite) y por Claude. Los 60 textos de chat
(30 ChatGPT, 30 Gemini) y los 6 docx de `eval_ia/ia/` son **solo test**.

Experimentos:
1. **Dejando un LLM fuera:** entrenar con Gemini-API y probar en Claude, y al revés.
2. **Modelo final:** entrenar con Gemini-API + Claude y probar en los textos de chat. ChatGPT es el
   **LLM nunca visto** en el entrenamiento. También se miden los humanos de test y las 20 UPeU de control.

Umbral: se elige en los humanos de **validación** para que den FP ≤ 5 %, y se reporta en los de **test**.
Criterio para desplegar: FP ≤ 5 % en humanos de test, recall ≥ 80 % en ChatGPT (nunca visto) **y**
recall ≥ 80 % en el LLM dejado fuera de los dos cruces Gemini ↔ Claude.

In [ ]:
!nvidia-smi -L
!pip -q install -U "transformers>=4.44" "accelerate>=0.33" scikit-learn pandas

## 1. Subir el dataset

In [ ]:
import os, zipfile
from google.colab import files

if not os.path.exists("segmentos.jsonl"):
    subido = files.upload()  # elige dataset_clasificador_ia.zip
    nombre = next(iter(subido))
    zipfile.ZipFile(nombre).extractall(".")
print(open("resumen.json", encoding="utf-8").read()[:1500])

In [ ]:
import pandas as pd
df = pd.read_json("segmentos.jsonl", lines=True)
print(len(df), "segmentos")
print(df.groupby(["origen", "particion", "solape"]).size().to_string())
# Ningún documento puede estar en dos particiones.
assert (df.groupby("documento")["particion"].nunique() == 1).all()

## 2. Modelo y entrenamiento

In [ ]:
import json, time, numpy as np, torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding, set_seed)

MAX_LENGTH = 384
EPOCHS = 3
SEED = 42
CANDIDATOS = ["PlanTL-GOB-ES/roberta-base-bne", "dccuchile/bert-base-spanish-wwm-cased"]

MODELO_BASE = None
for nombre in CANDIDATOS:
    try:
        tokenizer = AutoTokenizer.from_pretrained(nombre)
        AutoModelForSequenceClassification.from_pretrained(nombre, num_labels=2)
        MODELO_BASE = nombre
        break
    except Exception as exc:
        print("No carga", nombre, "->", exc)
print("Modelo base:", MODELO_BASE)

class Segmentos(Dataset):
    def __init__(self, frame):
        self.enc = tokenizer(frame["texto"].tolist(), truncation=True, max_length=MAX_LENGTH)
        self.labels = frame["etiqueta"].tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

class TrainerConPesos(Trainer):
    """Pérdida con pesos por clase (hay muchos más humanos que IA)."""
    def __init__(self, *a, pesos=None, **kw):
        super().__init__(*a, **kw)
        self.pesos = pesos
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=self.pesos.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def entrenar(train_df, salida):
    set_seed(SEED)
    modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=2)
    n = len(train_df); n_ia = int(train_df["etiqueta"].sum()); n_h = n - n_ia
    pesos = torch.tensor([n / (2 * n_h), n / (2 * n_ia)], dtype=torch.float)
    print(f"train: {n_h} humanos, {n_ia} IA, pesos {pesos.tolist()}")
    args = TrainingArguments(
        output_dir=salida, num_train_epochs=EPOCHS, learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        warmup_steps=max(1, int(0.1 * EPOCHS * -(-n // 16))), weight_decay=0.01, fp16=torch.cuda.is_available(),
        save_strategy="no", logging_steps=50, report_to=[], seed=SEED)
    trainer = TrainerConPesos(model=modelo, args=args, train_dataset=Segmentos(train_df),
                              data_collator=DataCollatorWithPadding(tokenizer), pesos=pesos)
    t0 = time.time(); trainer.train(); print(f"entrenado en {(time.time()-t0)/60:.1f} min")
    return trainer

def probabilidades(trainer, frame):
    if len(frame) == 0:
        return np.array([])
    pred = trainer.predict(Segmentos(frame))
    return torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()

## 3. Métricas

In [ ]:
from sklearn.metrics import roc_auc_score

def umbral_fp(prob_humanos_val, fp_max=0.05):
    """Menor umbral con el que, como mucho, el 5 % de los humanos de validación queda por encima."""
    return float(np.quantile(prob_humanos_val, 1 - fp_max, method="higher")) + 1e-6

def reporte(nombre, p_hum, p_ia, umbral, estilos=None):
    fila = {"experimento": nombre, "humanos": len(p_hum), "ia": len(p_ia), "umbral": round(umbral, 4)}
    if len(p_hum) and len(p_ia):
        y = np.r_[np.zeros(len(p_hum)), np.ones(len(p_ia))]
        fila["auroc"] = round(roc_auc_score(y, np.r_[p_hum, p_ia]), 4)
    fila["fp_humanos_%"] = round(100 * float((p_hum >= umbral).mean()), 2) if len(p_hum) else None
    fila["recall_ia_%"] = round(100 * float((p_ia >= umbral).mean()), 2) if len(p_ia) else None
    if estilos is not None and len(p_ia):
        for estilo in ["normal", "estudiante", "humanizado"]:
            m = np.asarray(estilos) == estilo
            if m.any():
                fila[f"recall_{estilo}_%"] = round(100 * float((p_ia[m] >= umbral).mean()), 2)
    return fila

RESULTADOS = []
base = lambda f: f[~f["solape"]]
hum = df[df["origen"] == "humano_corpus"]
H_TRAIN, H_VAL, H_TEST = (hum[hum["particion"] == p] for p in ["train", "val", "test"])
CONTROL = df[df["origen"] == "humano_control"]
API = {o: df[df["origen"] == f"{o}_api"] for o in ["gemini", "claude"]}
CHAT = df[df["particion"] == "test_chat"]

## 4. Dejando un LLM fuera (Gemini-API ↔ Claude)

In [ ]:
for entrena, prueba in [("gemini", "claude"), ("claude", "gemini")]:
    print(f"=== entrenar con {entrena}, probar en {prueba} ===")
    fuente = API[entrena]
    train_df = pd.concat([H_TRAIN, fuente[fuente["particion"] == "train"]])
    trainer = entrenar(train_df, f"out_{entrena}")
    p_val = probabilidades(trainer, H_VAL)
    umbral = umbral_fp(p_val)
    fuera = base(API[prueba])  # TODOS sus documentos, sin ventanas solapadas
    p_test_h = probabilidades(trainer, H_TEST)
    p_fuera = probabilidades(trainer, fuera)
    RESULTADOS.append(reporte(f"{entrena}->{prueba}", p_test_h, p_fuera, umbral, fuera["estilo"]))
    p_chat = probabilidades(trainer, CHAT)
    for origen in ["chatgpt_chat", "gemini_chat", "ia_docx"]:
        m = (CHAT["origen"] == origen).to_numpy()
        RESULTADOS.append(reporte(f"{entrena}->{origen}", p_test_h, p_chat[m], umbral, CHAT["estilo"][m]))
    del trainer; torch.cuda.empty_cache()
pd.DataFrame(RESULTADOS)

## 5. Modelo final (Gemini-API + Claude)

Se entrena con los humanos de train y **todos** los documentos de IA por API (train + val).
Los humanos de validación se reservan para fijar el umbral y los de test para medir FP: sin ellos
no habría forma honesta de medir falsos positivos.

In [ ]:
ia_api = pd.concat([API["gemini"], API["claude"]])
train_final = pd.concat([H_TRAIN, ia_api[(ia_api["particion"] == "train") | ~ia_api["solape"]]])
final = entrenar(train_final, "out_final")
p_val = probabilidades(final, H_VAL)
UMBRAL = umbral_fp(p_val)
p_test_h = probabilidades(final, H_TEST)
p_control = probabilidades(final, CONTROL)
p_chat = probabilidades(final, CHAT)
RESULTADOS.append(reporte("final->chat (60 + 6 docx)", p_test_h, p_chat, UMBRAL, CHAT["estilo"]))
for origen in ["chatgpt_chat", "gemini_chat", "ia_docx"]:
    m = (CHAT["origen"] == origen).to_numpy()
    RESULTADOS.append(reporte(f"final->{origen}", p_test_h, p_chat[m], UMBRAL, CHAT["estilo"][m]))
RESULTADOS.append(reporte("final->control UPeU (solo humanos)", p_control, np.array([]), UMBRAL))
tabla = pd.DataFrame(RESULTADOS)
tabla

In [ ]:
# Criterio para desplegar.
fuera = tabla[tabla["experimento"].isin(["gemini->claude", "claude->gemini"])]
final_chat = tabla[tabla["experimento"] == "final->chat (60 + 6 docx)"].iloc[0]
final_chatgpt = tabla[tabla["experimento"] == "final->chatgpt_chat"].iloc[0]
criterio = {
    "fp_humanos_test_final_%": final_chat["fp_humanos_%"],
    "recall_chatgpt_nunca_visto_%": final_chatgpt["recall_ia_%"],
    "recall_llm_fuera_min_%": float(fuera["recall_ia_%"].min()),
    "recall_chat_final_%": final_chat["recall_ia_%"],
}
criterio["desplegar"] = bool(
    criterio["fp_humanos_test_final_%"] <= 5
    and criterio["recall_chatgpt_nunca_visto_%"] >= 80
    and criterio["recall_llm_fuera_min_%"] >= 80
)
print(json.dumps(criterio, indent=1))

# % de segmentos sobre el umbral por documento de chat (como lo verá el sistema).
chat = CHAT.assign(prob=p_chat, ia=p_chat >= UMBRAL)
print(chat.groupby(["origen", "documento"])["ia"].mean().mul(100).round(1).groupby("origen").describe())

## 6. Guardar y descargar el modelo final

In [ ]:
import shutil
destino = "clasificador_ia"
final.save_model(destino)
tokenizer.save_pretrained(destino)
with open(f"{destino}/umbral.json", "w", encoding="utf-8") as f:
    json.dump({"modelo_base": MODELO_BASE, "max_length": MAX_LENGTH, "epocas": EPOCHS,
               "umbral": UMBRAL, "fp_objetivo": 0.05, "criterio": criterio,
               "resultados": RESULTADOS}, f, ensure_ascii=False, indent=1, default=float)
pd.DataFrame({"documento": CHAT["documento"], "origen": CHAT["origen"], "estilo": CHAT["estilo"],
              "prob": p_chat}).to_csv(f"{destino}/predicciones_chat.csv", index=False)
pd.DataFrame({"documento": H_TEST["documento"], "prob": p_test_h}).to_csv(
    f"{destino}/predicciones_humanos_test.csv", index=False)
shutil.make_archive("clasificador_ia", "zip", destino)
files.download("clasificador_ia.zip")